In [ ]:
import math
import random

cities = {0:(2,3), 1:(5,7), 2:(1,9), 3:(8,2), 4:(4,5)}

def distance(a, b):
    x1, y1 = cities[a]
    x2, y2 = cities[b]
    return math.sqrt((x1 - x2)**2 + (y1 - y2)**2)

def total_distance(route):
    size = len(route)
    total = 0
    for i in range(size-1):
        total += distance(route[i], route[i+1])
    total += distance(route[-1], route[0])
    return total

def init_population(size):
    population = []
    nodes = list(cities.keys())
    for _ in range(size):
        ind = nodes[:]
        random.shuffle(ind)
        population.append(ind)
    return population

def tournament_selection(pop, k=3):
    selected = random.sample(pop, k)
    return min(selected, key=total_distance)

def cycle_crossover(p1, p2):
    size = len(p1)
    def create_child(parent1, parent2):
        child = [None]*size
        cycles_done = [False]*size
        index = 0
        while not all(cycles_done):
            while cycles_done[index]:
                index += 1
            start = index
            while True:
                child[index] = parent1[index]
                cycles_done[index] = True
                val = parent2[index]
                index = parent1.index(val)
                if index == start:
                    break
        for i in range(size):
            if child[i] is None:
                child[i] = parent2[i]
        return child
    return create_child(p1,p2), create_child(p2,p1)

def pmx(p1, p2):
    size = len(p1)
    c1 = [None]*size
    c2 = [None]*size
    a = random.randint(0,size-2)
    b = random.randint(a+1,size-1)
    c1[a:b+1] = p1[a:b+1]
    c2[a:b+1] = p2[a:b+1]
    def fill(c,p):
        for i in range(size):
            if c[i] is None:
                gen = p[i]
                while gen in c:
                    gen = p[c.index(gen)]
                c[i] = gen
        return c
    return fill(c1,p2), fill(c2,p1)

def inversion_mutation(ind, rate=0.1):
    if random.random() < rate:
        i,j = sorted(random.sample(range(len(ind)),2))
        ind[i:j+1] = reversed(ind[i:j+1])
    return ind

def genetic_algo(pop_size, generations=50, crossover_type='cycle', mutation_rate=0.1):
    population = init_population(pop_size)
    best = min(population, key=total_distance)
    best_dist = total_distance(best)

    for gen in range(generations):
        new_pop = []
        while len(new_pop) < pop_size:
            p1 = tournament_selection(population)
            p2 = tournament_selection(population)

            if crossover_type=='cycle':
                c1, c2 = cycle_crossover(p1,p2)
            elif crossover_type=='pmx':
                c1, c2 = pmx(p1,p2)
            else:
                raise ValueError("Unknown crossover type")

            c1 = inversion_mutation(c1, mutation_rate)
            c2 = inversion_mutation(c2, mutation_rate)
            new_pop.extend([c1,c2])

        population = new_pop[:pop_size]
        current_best = min(population, key=total_distance)
        curr_distance = total_distance(current_best)
        if curr_distance < best_dist:
            best_dist = curr_distance
            best = current_best
    return best, best_dist

configs = [
    {'pop_size':5, 'crossover':'cycle', 'mutation':0.05},
    {'pop_size':10, 'crossover':'cycle', 'mutation':0.1},
    {'pop_size':15, 'crossover':'pmx', 'mutation':0.2}
]

for idx, cfg in enumerate(configs,1):
    best_route, best_dist = genetic_algo(
        pop_size=cfg['pop_size'],
        generations=100,
        crossover_type=cfg['crossover'],
        mutation_rate=cfg['mutation']
    )
    print(f"\nConfig {idx}: Pop={cfg['pop_size']}, Crossover={cfg['crossover']}, Mutation={cfg['mutation']}")
    print("Best Route:", best_route)
    print("Best Distance:", round(best_dist,2))